# 光电压 / 光电流分析
只读分析，不连接仪器。输入最终 JSON；默认仅选择整次采集和清理均成功的数据。仿真结果明确标记 SIMULATION。先运行 CLI simulate，再选择文件。
完整交互筛选和多维绘图请使用同目录 unified_plotting.ipynb。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd()/"src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT/"src"))
from attodry_control.photoresponse_analysis import load_results, export_results
RECORD = ROOT / "run_data" / "CHANGE_ME_completed.json"
# Rejected records remain on disk; this loader excludes them.
rows = load_results(RECORD)
print(f"{len(rows)} accepted conditions")
rows[:3]

## 明暗差随偏置变化
按光学设置和环境扫描点分别作图；实际温磁场读数保存在 rows 中。横轴为请求的源偏置，纵轴为采得的亮暗差；源设置不等于实际样品电流。无偏置模式以重复序号显示。差分按记录 method 使用对应单暗点或三个窗口的时间插值暗背景，不计算未经标定的 A/W 或 V/W。

In [ ]:
import json
import matplotlib.pyplot as plt
from collections import defaultdict
groups = defaultdict(list)
for row in rows:
    key = (json.dumps(row["requested_optical"], sort_keys=True), row["environment_index"])
    groups[key].append(row)
figures = []
for key, selected in groups.items():
    quantity = selected[0]["quantity"]
    mode = selected[0]["excitation"]
    bias_key = {"dc_current": "requested_current_a", "dc_voltage": "requested_voltage_v"}.get(mode)
    power_axis = bias_key is None and all(r.get("measured_power_w") is not None for r in selected)
    x = [r[bias_key] for r in selected] if bias_key else [r["measured_power_w"] for r in selected] if power_axis else list(range(len(selected)))
    scale, unit = (1e6, "uV") if quantity == "voltage_v" else (1e9, "nA")
    fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
    ax.scatter(x, [r["delta_"+quantity]*scale for r in selected])
    ax.set(xlabel={"dc_current":"Requested current (A)", "dc_voltage":"Requested voltage (V)"}.get(mode,"Measured monitor power (W)" if power_axis else "Repeated condition index"), ylabel=f"Light - dark ({unit})")
    prefix = "SIMULATION | " if selected[0]["simulated"] else ""
    optical = selected[0]["requested_optical"]
    parts = [f"Requested wavelength: {optical['wavelength_nm']:g} nm"] if optical.get("wavelength_nm") else ["Broadband"]
    if "current_a" in optical:
        parts.append(f"LD current: {optical['current_a']*1000:g} mA")
    else:
        parts.append(f"Initial source setting: {optical['source_level_pct']:g}%")
    prefix += "AUDIT rejected | " if any(not r["accepted"] for r in selected) else ""
    title = prefix + " | ".join(parts)
    if selected[0]["measurement_temperature_k"] is not None:
        from statistics import fmean
        actual_t = fmean(r["measurement_temperature_k"] for r in selected)
        actual_bx = fmean(r["measurement_bx_t"] for r in selected)
        actual_bz = fmean(r["measurement_bz_t"] for r in selected)
        title += f"\nMean readback T={actual_t:.4g} K, Bx={actual_bx:.4g} T, Bz={actual_bz:.4g} T"
    ax.set_title(title, fontsize=9)
    if bias_key is None and not power_axis:
        ax.set_xticks(range(len(selected)))
        ax.set_xlim(-.5, max(.5, len(selected)-.5))
    ax.grid(alpha=.25)
    figures.append(fig)
plt.show()

## 导出
显式设置新路径再运行，CSV 和 manifest 不覆盖已有文件。manifest 保存原始文件路径、SHA-256 和 accepted-only 选择规则；原始 JSON/JSONL/SQLite 保留。

In [ ]:
# EXPORT = ROOT / "analysis_output" / "photoresponse_results.csv"
# manifest = export_results(RECORD, EXPORT)
# for index, fig in enumerate(figures):
#     fig.savefig(EXPORT.with_name(f"photoresponse_{index}.png"), dpi=300)